In [171]:
# FAG_Extract_Cemeteries.py
# 6/16/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time_pause,sleep_range,pause_count
    if call_count%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}\n")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 


def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return capture_json,total_cemeteries

    
def extract_cemetery_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['data']['cemeteries']['cemeteries'])
    collection_columns_set=set(list(collection_df))
    
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    collection_df=collection_df[field_list]
    return collection_df




def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

def extract_cemetery_names(collection_df):   
    global location_fields,cemetery_fields
    cemetery_frame_df=pd.DataFrame([])   
    for index, row in collection_df.iterrows():    
        cemetery_data=row['locations']
        cemetery_data_df=pd.json_normalize(cemetery_data)
        cemetery_frame_temp_df=pd.DataFrame([])
        cemetery_data_df=cemetery_data_df[cemetery_fields]
        for index2 in range(len(cemetery_data_df)):
            cemetery_frame_temp_df[location_fields]=pd.DataFrame(collection_df[location_fields].iloc[index]).T
            cemetery_frame_temp_df[cemetery_fields]=cemetery_data_df.iloc[index2].values
            cemetery_frame_df=pd.concat([cemetery_frame_df,cemetery_frame_temp_df])   
    return cemetery_frame_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'ancestry-clientpath': 'findagrave-frontend',
  'apollographql-client-name': 'findagrave-cemetery-searchresults',
  'content-type': 'application/json',
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'
}

blank_line='                                                                                                                                                                                                 '  


maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0


start_time=datetime.datetime.now()
event_time=start_time


process_name='Extract FAG Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileStates="Benjamin_States"
fileCounties="Benjamin_Counties"
fileOutput="Benjamin_Cemeteries"

if append==1:fileIn='Benjamin_Counties_X'

fileInCounties=DirMain+"Data\\"+fileCounties+".csv"
fileInStates=DirMain+"Data\\"+fileStates+".csv"

fileOut=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"OutputData\\Cemeteries\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_Interim.csv"

county_info_df=pd.read_csv(fileInCounties)
state_info_df=pd.read_csv(fileInStates)

if append==1:print('**** Appending ****')    
if append!=1:cemetery_df=pd.DataFrame([])  
county_df_header=['state_name','state_id','county_id','county_name','county_lat','county_lon']

field_list=['id', 'locations', 'memorialCount', 'coordinates.lat', 'coordinates.lon']
field_list_set=set(field_list)

url_state="https://www.findagrave.com/orc/graphql?operationName=cemeterySearch&variables=%7B%22search%22%3A%7B%22name%22%3A%22%22%2C%22containingLocationId%22%3A%22"
url_skip="%22%2C%22from%22%3A"
url_suffix="%2C%22size%22%3A100%2C%22orderBy%22%3A%22name%22%7D%7D&extensions=%7B%22persistedQuery%22%3A%7B%22version%22%3A1%2C%22sha256Hash%22%3A%22068e8502a01fbcd4febfb1fe17b0d09d1b76478b7c9e3b3815a76fc61ac33acd%22%7D%7D"
sleep_range=[5,10]
sleep_time_pause=1*60
pause_count=100
cemetery_count=0
get_counter=0
state_count=0
total_state_count=len(state_info_df)
cemetery_name_count=0
cemetery_df=pd.DataFrame([])
for index, row in state_info_df.iterrows():
    state_count+=1
    state_name=row['state_name']
    state_id=row['state_id']
    cemetery_name_count+=1
    cemetery_frame_count=0
    cemetery_count=10000
    cemetery_count_offset=0
    print(f"\n *** {process_name:<20}  • State: *{state_name} - {state_id} * ")
    while cemetery_frame_count<cemetery_count:        
        url=url_state+state_id+url_skip+str(cemetery_frame_count)+url_suffix
        capture_json,cemetery_count=get_total_cemeteries(url)
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time) 
        if cemetery_count>10000:
            county_id_df=county_info_df['county_id'][county_info_df['state_id']==state_id]
            total_county_count=len(county_id_df)
            county_count=0
            for county_id in county_id_df:
                cemetery_count=10000
                cemetery_frame_count=0
                county_count+=1
                while cemetery_frame_count<cemetery_count:                 
                    url=url_state+county_id+url_skip+str(cemetery_frame_count)+url_suffix
                    capture_json,cemetery_count=get_total_cemeteries(url)
                    collection_df=extract_cemetery_data(capture_json)
                    cemetery_frame_df=extract_cemetery_names(collection_df)
                    cemetery_df=pd.concat([cemetery_df,cemetery_frame_df])
                    cemetery_df.to_csv(fileOutInt,mode="w", index=False,header=True)
                    cemetery_frame_count+=100   
                    print(f" {blank_line}", end='\r')
                    print(f"   .... {process_name:<20} **County Processing**   *{state_name}* {state_id}-{county_id} * • Cemetery Count: {cemetery_frame_count}/{cemetery_count} "\
                                      +f"• States Processed: {state_count}/{total_state_count} • Counties Processed: {county_count}/{total_county_count} ******* " , end='\r')                       
                    get_counter+=1
                    sleep_time=random_sleep(get_counter)       
                    time.sleep(sleep_time)         
            print(f"  **** {process_name:<20} **County Processing**   *{state_name}* {state_id}-{county_id} * • Cemetery Count: {cemetery_count} "\
                                      +f"• States Processed: {state_count}/{total_state_count} • Counties Processed: {county_count}/{total_county_count} ******* ")                      

        collection_df=extract_cemetery_data(capture_json)
        cemetery_frame_df=extract_cemetery_names(collection_df)
        cemetery_df=pd.concat([cemetery_df,cemetery_frame_df])
        cemetery_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time)       
        cemetery_frame_count+=100
        print(f" {blank_line}", end='\r')
        print(f"   .... {process_name:<20}   *{state_name}* {state_id} * • Cemetery Count: {cemetery_frame_count}/{cemetery_count} • States Processed: {state_count}/{total_state_count} ******* " , end='\r')             
    print(f"  **** {process_name:<20}   *{state_name}-{state_id} * • Cemetery Count: {cemetery_count} • States Processed: {state_count}/{total_state_count} ******* ")              



cemetery_df.to_csv(fileOut,mode="w", index=False,header=True)
cemetery_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Extract FAG Cemeteries   *** , Time: 2025-06-16 16:49:01.900066  **


 *** Extract FAG Cemeteries   • State: * Alabama - state_3 * 
			Sleeping for 60 seconds • Sleeping till: 06/16/2025 17:03:40ma-state_3-county_64 * • Cemetery Count: 100/307 • States Processed: 1/52 • Counties Processed: 39/67 *******                      
  **** Extract FAG Cemeteries    * Alabama-state_3 * • Cemetery Count: 176 • States Processed: 1/52 ******* ****                                                                                  

 *** Extract FAG Cemeteries   • State: * Alaska - state_2 * 
  **** Extract FAG Cemeteries    * Alaska-state_2 * • Cemetery Count: 541 • States Processed: 2/52 ******* ****                                                                                   

 *** Extract FAG Cemeteries   • State: * Arizona - state_5 * 
			Sleeping for 60 seconds • Sleeping till: 06/16/2025 17:18:13
  **** Extract FAG Cemeteries    * Arizona-state_5 * • Cemetery Count: 1315 • 

In [ ]:
url_state+county_id+url_skip+str(cemetery_frame_count)+url_suffix

In [162]:
url

'https://www.findagrave.com/orc/graphql?operationName=cemeterySearch&variables=%7B%22search%22%3A%7B%22name%22%3A%22%22%2C%22containingLocationId%22%3A%22state_3%22%2C%22from%22%3A200%2C%22size%22%3A100%2C%22orderBy%22%3A%22name%22%7D%7D&extensions=%7B%22persistedQuery%22%3A%7B%22version%22%3A1%2C%22sha256Hash%22%3A%22068e8502a01fbcd4febfb1fe17b0d09d1b76478b7c9e3b3815a76fc61ac33acd%22%7D%7D'

In [172]:
cemetery_df.to_csv(fileOut,mode="w", index=False,header=True)
cemetery_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)




Uploaded To:  C:\Users\jrozm\OneDrive\Benjamin\OutputData\Cemeteries\Benjamin_Cemeteries.csv


In [ ]:
cemetery_data_temp_df